In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Analisis_Scraping") \
    .config("spark.mongodb.read.connection.uri", "mongodb://bigdata_mongodb:27017") \
    .config("spark.mongodb.read.database", "proyecto_bigdata") \
    .config("spark.mongodb.read.collection", "datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()


In [15]:
# Definir el orden deseado de columnas
columnas = ["_id", "Grupo", "disponibilidad", "pagina", "precio_gbp", "rating", "titulo", "url"]

# Crear un nuevo DataFrame con ese orden
df_reordenado = df.select(*columnas)

# Mostrar el esquema y algunos registros
df_reordenado.printSchema()
df_reordenado.show(5)


root
 |-- _id: string (nullable = true)
 |-- Grupo: string (nullable = true)
 |-- disponibilidad: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- precio_gbp: double (nullable = true)
 |-- rating: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- url: string (nullable = true)

+--------------------+--------------------+--------------+------+----------+------+--------------------+--------------------+
|                 _id|               Grupo|disponibilidad|pagina|precio_gbp|rating|              titulo|                 url|
+--------------------+--------------------+--------------+------+----------+------+--------------------+--------------------+
|69e22aa1592ad4126...|G1_ecommerce&precios|      In stock|     1|     51.77|     3|A Light in the Attic|https://books.tos...|
|69e22aa1592ad4126...|G1_ecommerce&precios|      In stock|     1|     53.74|     1|  Tipping the Velvet|https://books.tos...|
|69e22aa1592ad4126...|G1_ecommerce&precios|      In s

In [16]:
df.printSchema()

root
 |-- Grupo: string (nullable = true)
 |-- _id: string (nullable = true)
 |-- disponibilidad: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- precio_gbp: double (nullable = true)
 |-- rating: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- url: string (nullable = true)



In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Analisis_Scraping") \
    .config("spark.mongodb.read.connection.uri", "mongodb://bigdata_mongodb:27017") \
    .config("spark.mongodb.read.database", "proyecto_bigdata") \
    .config("spark.mongodb.read.collection", "datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

# Verificar esquema
df.printSchema()

# Filtrar registros: titulo no nulo y precio mayor a 0
df_filtrado = df.filter((col("titulo").isNotNull()) & (col("precio_gbp") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros válidos:", df_filtrado.count())

df_filtrado.show(10)


root
 |-- Grupo: string (nullable = true)
 |-- _id: string (nullable = true)
 |-- disponibilidad: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- precio_gbp: double (nullable = true)
 |-- rating: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- url: string (nullable = true)

PASO 2: Limpieza completada.
Registros originales: 200
Registros válidos: 200
+--------------------+--------------------+--------------+------+----------+------+--------------------+--------------------+
|               Grupo|                 _id|disponibilidad|pagina|precio_gbp|rating|              titulo|                 url|
+--------------------+--------------------+--------------+------+----------+------+--------------------+--------------------+
|G1_ecommerce&precios|69e22aa1592ad4126...|      In stock|     1|     51.77|     3|A Light in the Attic|https://books.tos...|
|G1_ecommerce&precios|69e22aa1592ad4126...|      In stock|     1|     53.74|     1|  Tipping the Velv

In [18]:
df.select("titulo", "precio_gbp").show()

+--------------------+----------+
|              titulo|precio_gbp|
+--------------------+----------+
|A Light in the Attic|     51.77|
|  Tipping the Velvet|     53.74|
|          Soumission|      50.1|
|       Sharp Objects|     47.82|
|Sapiens: A Brief ...|     54.23|
|     The Requiem Red|     22.65|
|The Dirty Little ...|     33.34|
|The Coming Woman:...|     17.93|
|The Boys in the B...|      22.6|
|     The Black Maria|     52.15|
|Starving Hearts (...|     13.99|
|Shakespeare's Son...|     20.66|
|         Set Me Free|     17.46|
|Scott Pilgrim's P...|     52.29|
|Rip it Up and Sta...|     35.02|
|Our Band Could Be...|     57.25|
|                Olio|     23.88|
|Mesaerion: The Be...|     37.59|
|Libertarianism fo...|     51.33|
|It's Only the Him...|     45.17|
+--------------------+----------+
only showing top 20 rows



In [19]:
df.filter(df["precio_gbp"] > 50).show()

+--------------------+--------------------+--------------+------+----------+------+--------------------+--------------------+
|               Grupo|                 _id|disponibilidad|pagina|precio_gbp|rating|              titulo|                 url|
+--------------------+--------------------+--------------+------+----------+------+--------------------+--------------------+
|G1_ecommerce&precios|69e22aa1592ad4126...|      In stock|     1|     51.77|     3|A Light in the Attic|https://books.tos...|
|G1_ecommerce&precios|69e22aa1592ad4126...|      In stock|     1|     53.74|     1|  Tipping the Velvet|https://books.tos...|
|G1_ecommerce&precios|69e22aa1592ad4126...|      In stock|     1|      50.1|     1|          Soumission|https://books.tos...|
|G1_ecommerce&precios|69e22aa2592ad4126...|      In stock|     1|     54.23|     5|Sapiens: A Brief ...|https://books.tos...|
|G1_ecommerce&precios|69e22aa3592ad4126...|      In stock|     1|     52.15|     1|     The Black Maria|https://books.

In [20]:
df.groupBy("grupo").count().show()

+--------------------+-----+
|               grupo|count|
+--------------------+-----+
|G1_ecommerce&precios|  200|
+--------------------+-----+



In [22]:
from pyspark.sql import functions as F

# 2. Transformación y Agregación por GRUPO
# Esto permite comparar qué grupos son más caros en promedio
reporte_grupo = df.groupBy("Grupo").agg(
    F.count("titulo").alias("cantidad_libros"),
    F.avg("precio_gbp").alias("precio_promedio"),
    F.min("precio_gbp").alias("precio_minimo"),
    F.max("precio_gbp").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANÁLISIS DE MERCADO POR GRUPO:")
reporte_grupo.show()


ANÁLISIS DE MERCADO POR GRUPO:
+--------------------+---------------+------------------+-------------+-------------+
|               Grupo|cantidad_libros|   precio_promedio|precio_minimo|precio_maximo|
+--------------------+---------------+------------------+-------------+-------------+
|G1_ecommerce&precios|            200|34.560700000000004|        10.16|        58.11|
+--------------------+---------------+------------------+-------------+-------------+



In [12]:
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- disponibilidad: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- precio_gbp: double (nullable = true)
 |-- rating: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- url: string (nullable = true)

+--------------------+--------------+------+----------+------+--------------------+--------------------+
|                 _id|disponibilidad|pagina|precio_gbp|rating|              titulo|                 url|
+--------------------+--------------+------+----------+------+--------------------+--------------------+
|69e22aa1592ad4126...|      In stock|     1|     51.77|     3|A Light in the Attic|https://books.tos...|
|69e22aa1592ad4126...|      In stock|     1|     53.74|     1|  Tipping the Velvet|https://books.tos...|
|69e22aa1592ad4126...|      In stock|     1|      50.1|     1|          Soumission|https://books.tos...|
|69e22aa2592ad4126...|      In stock|     1|     47.82|     4|       Sharp Objects|https:/

In [23]:
from pyspark.sql import functions as F

# Ordenar por precio descendente y seleccionar las columnas relevantes
fila = df.orderBy(F.desc("precio_gbp")).select(
    "titulo", "Grupo", "pagina", "precio_gbp"
).first()

print("Producto más caro:", fila["titulo"])
print("Grupo:", fila["Grupo"])
print("Página de captura:", fila["pagina"])
print("Precio GBP:", fila["precio_gbp"])


Producto más caro: The Death of Humanity: and the Case for Life
Grupo: G1_ecommerce&precios
Página de captura: 4
Precio GBP: 58.11
